In [1]:
import torch
from datasets import load_dataset

/home/elena/miniconda/envs/emcomm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
wg = load_dataset('facebook/winoground')
wg

DatasetDict({
    test: Dataset({
        features: ['id', 'image_0', 'image_1', 'caption_0', 'caption_1', 'tag', 'secondary_tag', 'num_main_preds', 'collapsed_tag'],
        num_rows: 400
    })
})

In [3]:
from torchvision import models, transforms

# Load ResNet model and set to eval mode
resnet = models.resnet152(pretrained=True)
resnet.eval()

# Define image preprocessing
preprocess = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

def image_to_features(example):
    # Convert PIL image to tensor and preprocess
    image = example#["image"]
    input_tensor = preprocess(image).unsqueeze(0).to("cuda" if torch.cuda.is_available() else "cpu")
    resnet.to("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        features = resnet(input_tensor)
    # return {"image_tensor": features.cpu().squeeze().numpy()}
    return features.cpu().squeeze().numpy()

/home/elena/miniconda/envs/emcomm/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/elena/miniconda/envs/emcomm/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [4]:
new_ds =wg['test'].map(
    lambda example: {
        # "captions": example["captions"],
        "features_0": torch.tensor(image_to_features(example["image_0"])),
        "features_1": torch.tensor(image_to_features(example["image_1"]))
    },
    # remove_columns=[col for col in original_ds.column_names if col not in ["captions"]],
    # num_proc=24
)

Map: 100%|██████████| 400/400 [11:48<00:00,  1.77s/ examples] 


In [6]:
new_ds

Dataset({
    features: ['id', 'image_0', 'image_1', 'caption_0', 'caption_1', 'tag', 'secondary_tag', 'num_main_preds', 'collapsed_tag', 'features_0', 'features_1'],
    num_rows: 400
})

In [7]:
new_ds.save_to_disk("../../../datasets/winoground_features_resnet_152")

Saving the dataset (3/3 shards): 100%|██████████| 400/400 [00:13<00:00, 30.65 examples/s] 


In [8]:
new_ds[0]

{'id': 0,
 'image_0': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1920x1280>,
 'image_1': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1920x1280>,
 'caption_0': 'an old person kisses a young person',
 'caption_1': 'a young person kisses an old person',
 'tag': 'Adjective-Age',
 'secondary_tag': '',
 'num_main_preds': 1,
 'collapsed_tag': 'Relation',
 'features_0': [-1.8375893831253052,
  -2.962782382965088,
  -0.6948517560958862,
  -2.7026243209838867,
  -0.21987564861774445,
  -1.2815124988555908,
  0.6708372235298157,
  4.414561748504639,
  0.4696517288684845,
  -0.9247156381607056,
  -3.178877115249634,
  -3.9031689167022705,
  -3.4653639793395996,
  -0.9091799259185791,
  -2.182065725326538,
  -1.106176495552063,
  -2.5013415813446045,
  -1.0933120250701904,
  0.3796710968017578,
  0.3850782513618469,
  1.5226445198059082,
  1.5476874113082886,
  -0.5388423204421997,
  1.1215828657150269,
  -1.5215656757354736,
  -3.942857503890991,
  -4.414251804351807,
  -2.635